<div style="background-color: #1e1e2e; padding: 20px; border-radius: 10px; border-left: 5px solid #a6e3a1;">
    <h1 style="color: #cba6f7; margin-top: 0; font-family: sans-serif;">🚀 Application Deployment: Gradio App</h1>
    <h3 style="color: #bac2de; margin-bottom: 0; font-family: sans-serif; font-weight: normal;">Lung Cancer Detection from CT Scans Dataset</h3>
    <hr style="border: 1px solid #313244; margin-top: 15px; margin-bottom: 15px;">
    <p style="color: #a6adc8; margin-bottom: 0; font-family: sans-serif;">
        This notebook launches an interactive web application using Gradio. It loads the trained ResNet-18 model and provides a user-friendly interface for uploading CT scans and predicting the presence of lung cancer in real-time.
    </p>
</div>


### 📦 1. Importing Dependencies
First, we import the necessary libraries. We will use `gradio` for the web interface, `PIL` for image handling, and `torch`/`torchvision` to load our deep learning model.


In [ ]:
import gradio as gr
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights


### ⚙️ 2. Configuration & Model Initialization
We configure our classes (Positive and Negative) and initialize the ResNet-18 model architecture. We then load our trained weights and set the model to evaluation mode (`model.eval()`).


In [ ]:
device = torch.device("cpu")
num_classes = 2

classes = ["Postive", "Negative"]

model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)
from safetensors.torch import load_model
load_model(model, "Model/lung_cancer_cnn_v1.safetensors")
model = model.to(device)
model.eval()


### 🖼️ 3. Image Preprocessing
Before feeding images to the model, we must preprocess them. This involves resizing to `224x224`, converting to tensors, and normalizing the pixels based on ImageNet statistics.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


### 🧠 4. Prediction Logic
The `predict` function takes an uploaded image, transforms it, and performs a forward pass through the model to retrieve confidence probabilities for each class.


In [ ]:
def predict(image):
    img = Image.fromarray(image).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.softmax(output, dim=1)[0]
        predicted_idx = torch.argmax(probabilities).item()
    label = classes[predicted_idx]
    confidence = {classes[i]: float(probabilities[i]) for i in range(num_classes)}
    return label, confidence


### 🌐 5. Building the Gradio Interface
We build an intuitive user interface using `gr.Blocks`. It features a title, a brief description, an image upload module, prediction outputs, and pre-loaded examples for quick testing.


In [ ]:
example_images = [
    ["Dataset/Negative/Normal case (1).jpg"],
    ["Dataset/Negative/Normal case (5).jpg"],
    ["Dataset/Positive/Bengin case (1).jpg"],
    ["Dataset/Positive/Bengin case (10).jpg"],
    ["Dataset/Positive/Malignant case (1).jpg"],
    ["Dataset/Positive/Malignant case (5).jpg"],
]

with gr.Blocks(title="Lung Cancer Detection") as demo:
    gr.Markdown(
        """
        # 🫁 Lung Cancer Detection
        Upload a lung CT scan image to detect whether it shows signs of **cancer** or is **normal**.
        The model uses a ResNet-18 CNN trained on lung CT scan data.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(label="Upload CT Scan Image", type="numpy")
            predict_btn = gr.Button("🔍 Analyze", variant="primary")

        with gr.Column(scale=1):
            label_output = gr.Label(label="Prediction")
            confidence_output = gr.Label(label="Confidence Scores", num_top_classes=num_classes)

    gr.Examples(
        examples=example_images,
        inputs=image_input,
        label="Example CT Scan Images"
    )

    predict_btn.click(
        fn=predict,
        inputs=image_input,
        outputs=[label_output, confidence_output]
    )


### 🚀 6. Launch the App
Finally, we launch the Gradio app. By running `demo.launch()`, a local server will start and you can access the interface in your browser.


In [ ]:
if __name__ == "__main__":
    demo.launch()


### 🎉 Conclusion
The ResNet-18 model was successfully integrated into a Gradio web application. This provides a simple and effective interface for classifying CT scans as either showing signs of cancer or being normal. This application can serve as a helpful tool for preliminary screening.
